# Large signal: three ways to get a wiggle onto the genome

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/13_large_wiggle.ipynb)

Coverage, conservation, methylation, a ChIP fold-change — quantitative signal is the data type that gets big fastest, because there's a value for every base or every bin. This notebook lays the three routes side by side and measures each, so you can pick by size rather than by guess.

| | how it travels | good to |
|---|---|---|
| `add_features` | every point inlined as JSON | ~100k points |
| bigWig + `add_local_file` | the file crosses once, then byte ranges | tens of MB |
| recompute per region | only what's on screen, every pan | **unlimited** |

Throughout, `signal` stands in for whatever your pipeline produced — see [05](05_bam_coverage.ipynb) for real pysam depth and [06](06_popgen_selection.ipynb) for a real scan.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy pyBigWig

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## The signal

Binned values along hg38 chr1 — the shape of a coverage track.

In [ ]:
import numpy as np

CHROM, CHROM_LEN = "1", 248_956_422
BIN = 100
rng = np.random.default_rng(0)
starts = np.arange(0, CHROM_LEN - BIN, BIN, dtype=np.int64)
signal = rng.gamma(2.0, 3.0, starts.size).astype(np.float32)
print(f"{starts.size:,} bins at {BIN}bp across chr1")

## 1. Inline it — `add_features`

A **`score`** column is what makes this a wiggle rather than boxes: the track comes back as a `QuantitativeTrack` with a value axis and autoscaling. (That's JBrowse's own name for the plotted value, so call the column `score`, not `depth` or `signal`.)

Every point is serialized into the widget's state, so this is priced per point — fine for a region, hopeless for a chromosome.

In [ ]:
import json

from jbrowse_anywidget import features_track


def rows(start_arr, value_arr):
    return [
        {"refName": CHROM, "start": int(s), "end": int(s) + BIN, "score": round(float(v), 2)}
        for s, v in zip(start_arr, value_arr)
    ]


per_point = len(json.dumps(features_track(rows(starts[:5000], signal[:5000])))) / 5000
print(f"inlined: ~{per_point * starts.size / 1e6:,.0f} MB for the whole chromosome")

Too much. But for a **window** it's exactly right — one call, no file, and it renders as a real wiggle:

In [ ]:
from jbrowse_anywidget import LinearGenomeView

view = LinearGenomeView(assembly="hg38", location="1:1,000,000..1,200,000")
window = (starts >= 1_000_000) & (starts < 1_200_000)
view.add_features(rows(starts[window], signal[window]), name="signal (inlined window)")
view

## 2. Write a bigWig — `add_local_file`

bigWig is the format built for this. It stores **precomputed zoom levels**, so viewing a whole chromosome reads a summary instead of every underlying point, and it's indexed, so any region is a seek. `add_local_file` pushes it into the browser once and JBrowse reads byte ranges out of it — no web server.

In [ ]:
import os

import pyBigWig

bw = pyBigWig.open("signal.bw", "w")
bw.addHeader([(CHROM, CHROM_LEN)])
CHUNK = 2_000_000  # addEntries is happier in batches
for i in range(0, starts.size, CHUNK):
    s = starts[i : i + CHUNK]
    bw.addEntries(
        [CHROM] * s.size,
        s.tolist(),
        ends=(s + BIN).tolist(),
        values=signal[i : i + CHUNK].astype(float).tolist(),
    )
bw.close()
size = os.path.getsize("signal.bw") / 1e6
print(f"bigWig: {size:.0f} MB, zoom levels included")

In [ ]:
view.add_track(view.add_local_file("signal.bw"))
view.location = "1"  # whole chromosome: served from a zoom level
view

Zoom in and the view switches to the underlying data automatically. The whole file crossed the comm once, though — at 10 bp bins rather than 100 this same chromosome is ~200 MB, and a genome is ~3 GB. That's where this route stops.

## 3. Recompute per region

The observation that removes the ceiling: a wiggle is only ever drawn at **screen resolution** — a couple of thousand bins across the view, however much data is underneath. So bin in the kernel for the visible window only, and the payload stops depending on the size of the data entirely.

The data never moves. It can be an array, a zarr store, a database, a file on a cluster the browser can't reach — anything Python can slice.

In [ ]:
import re

SCREEN_BINS = 1500


def parse_loc(loc):
    m = re.match(r"^\s*([^:\s]+)\s*:\s*([\d,]+)\s*\.\.\s*([\d,]+)", loc or "")
    return (int(m[2].replace(",", "")), int(m[3].replace(",", ""))) if m else None


def render_window(start, end):
    # ceiling division, so SCREEN_BINS is a ceiling not a target
    step = max(BIN, -(-(end - start) // SCREEN_BINS))
    edges = np.arange(start, end, step, dtype=np.int64)
    # mean of the underlying bins in each screen bin; the last one runs
    # to the end of the window rather than to its own start
    idx = np.searchsorted(starts, np.r_[edges, end])
    values = [
        float(signal[a:b].mean()) if b > a else 0.0
        for a, b in zip(idx, idx[1:])
    ]
    live.tracks = []  # replace the previous window
    live.add_features(
        [{"refName": CHROM, "start": int(s), "end": int(s) + step, "score": round(v, 2)}
         for s, v in zip(edges, values)],
        name="signal (recomputed for this view)",
        track_id="live",
    )


def on_location(change):
    region = parse_loc(change["new"])
    if region:
        render_window(*region)


live = LinearGenomeView(assembly="hg38", location="1:1,000,000..1,200,000")
live.observe(on_location, "location")
render_window(1_000_000, 1_200_000)
live

Pan or zoom and the kernel rebins for the new window. Each update is a fixed ~100-200 KB whether the data behind it is a megabyte or a terabyte:

In [ ]:
for span in (10_000, 1_000_000, CHROM_LEN):
    step = max(BIN, span // SCREEN_BINS)
    n = min(SCREEN_BINS, span // step)
    payload = len(json.dumps(features_track(
        [{"refName": CHROM, "start": int(i * step), "end": int((i + 1) * step), "score": 1.23}
         for i in range(n)]
    )))
    print(f"{span:>12,} bp window -> {payload / 1e3:5.0f} KB")

## Choosing

- **A region you already have in memory** → `add_features` with a `score` column. One call, no file.
- **A whole chromosome or genome you want to browse freely** → write a bigWig and `add_local_file`. Real zoom levels, real seeking, and the track keeps working if you later host the file instead.
- **Bigger than that, or not a file at all** → recompute per region. Constant cost, unlimited data, at the price of a round trip on every pan.

The three compose: a bigWig for the overview and a recomputed track for something expensive you only want for the visible window is a perfectly good pairing.